In [4]:
%%capture
!pip install tensorflow
!pip install keras
!pip install opencv-python
!pip install kagglehub

In [17]:
%%capture
import os
import numpy as np
import cv2
import kagglehub
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import preprocessing
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, UpSampling2D, Conv2D, BatchNormalization, Activation, LeakyReLU, Dropout, ZeroPadding2D, Flatten


In [3]:
print("Tensroflow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

Tensroflow Version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [13]:
image_folder = "/root/.cache/kagglehub/datasets/karnikakapoor/art-portraits/versions/3/Portraits/Portraits" #Don't forget to update it after downlading the DS
image_size = (64,64)
batch_size = 32
latent_dim = 100

#Dataset Downloading from Kaggel

In [6]:
print("Downloading art-portraits dataset...")
path = kagglehub.dataset_download("karnikakapoor/art-portraits")
print("Path to dataset files:", path)

100%|██████████| 3.20G/3.20G [01:25<00:00, 40.3MB/s]


Path to dataset files: /root/.cache/kagglehub/datasets/karnikakapoor/art-portraits/versions/3


In [9]:
def preprocess_dataset(folder, image_size, batch_size):
    """
    Preprocesses an image dataset from a directory.

    Args:
        folder (str): The path to the folder containing the images.
        image_size (tuple): A tuple of two integers representing the desired
                            image size (height, width).
        batch_size (int): The number of images per batch.

    Returns:
        tf.data.Dataset: A TensorFlow dataset containing the preprocessed images.
                         Pixel values are scaled to the range [-1.0, 1.0].
    """
    dataset = preprocessing.image_dataset_from_directory(
        folder,
        label_mode=None,  # No labels needed for generative tasks
        image_size=image_size,
        batch_size=batch_size)

    # Scale image pixel values from [0, 255] to [-1.0, 1.0]
    dataset = dataset.map(lambda x: (x/127.5)-1.0)
    return dataset

In [10]:
dataset = preprocess_dataset(
    folder=image_folder,
    image_size= image_size,
    batch_size=batch_size)

for batch in dataset.take(1):
    print(f"Batch shape: {batch.shape}")

Found 4117 files.
Batch shape: (32, 64, 64, 3)


## Building Generative Model from the Scratch

In [15]:

def build_generator():
    """
    Builds the generator model for a Generative Adversarial Network (GAN).

    The generator takes a latent vector (random noise) as input and
    outputs a generated image.

    Returns:
        tf.keras.models.Sequential: The generator model.
    """
    generator = Sequential()
    generator.add(Dense(4*4*256, activation="relu", input_dim=latent_dim))
    generator.add(Reshape((4,4,256)))
    generator.add(UpSampling2D())
    generator.add(Conv2D(256, kernel_size=3, padding="same"))
    generator.add(BatchNormalization(momentum=0.8))
    generator.add(Activation("relu"))
    generator.add(UpSampling2D())
    generator.add(Conv2D(256, kernel_size=3, padding="same"))
    generator.add(BatchNormalization(momentum=0.8))
    generator.add(Activation("relu"))
    generator.add(UpSampling2D())
    generator.add(Conv2D(256, kernel_size=3, padding="same"))
    generator.add(BatchNormalization(momentum=0.8))
    generator.add(Activation("relu"))
    generator.add(UpSampling2D())
    generator.add(Conv2D(256, kernel_size=3, padding="same"))
    generator.add(BatchNormalization(momentum=0.8))
    generator.add(Activation("relu"))
    generator.add(Conv2D(3, kernel_size=3, padding="same"))
    generator.add(Activation("tanh"))

    return generator


In [16]:
generator = build_generator()
generator.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 4096)           │       413,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_2 (UpSampling2D)  │ (None, 32, 32, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 32, 32, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_3 (UpSampling2D)  │ (None, 64, 64, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 64, 64, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 64, 3)      │         6,915 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 64, 64, 3)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,785,027 (10.62 MB)

 Trainable params: 2,782,979 (10.62 MB)

 Non-trainable params: 2,048 (8.00 KB)

## Building Discriminative Model from Scratch

In [18]:
def build_discriminator():
    """
    Builds the discriminator model for a Generative Adversarial Network (GAN).

    The discriminator takes an image (real or generated) as input and
    outputs a probability indicating whether the image is real or fake.

    Returns:
        tf.keras.models.Sequential: The discriminator model.
    """
    discriminator = Sequential()

    # Input layer: Takes a 64x64x3 image as input.
    # Applies a 2D convolutional layer with 32 filters, a 3x3 kernel,
    # a stride of 2 (downsampling the image by half), and same padding.
    discriminator.add(Conv2D(32, kernel_size=3, strides=2, input_shape=(64,64,3), padding="same"))
    # Applies the Leaky ReLU activation function with a small alpha (slope) for negative values.
    discriminator.add(LeakyReLU(alpha=0.2))
    # Applies dropout for regularization, randomly setting a fraction of inputs to 0.
    discriminator.add(Dropout(0.25))

    # Convolutional block 1:
    # Applies a 2D convolutional layer with 64 filters, a 3x3 kernel,
    # a stride of 2 (downsampling), and same padding.
    discriminator.add(Conv2D(64, kernel_size=3, strides=2, padding="same"))
    # Adds zero padding to the feature map. This can be used to handle
    # spatial dimensions after strided convolutions.
    discriminator.add(ZeroPadding2D(padding=((0,1),(1,0))))
    # Applies Batch Normalization to stabilize training.
    discriminator.add(BatchNormalization(momentum=0.8))
    # Applies the Leaky ReLU activation function.
    discriminator.add(LeakyReLU(alpha=0.2))
    # Applies dropout for regularization.
    discriminator.add(Dropout(0.25))

    # Convolutional block 2:
    # Applies a 2D convolutional layer with 128 filters, a 3x3 kernel,
    # a stride of 2 (downsampling), and same padding.
    discriminator.add(Conv2D(128, kernel_size=3, strides=2, padding="same"))
    # Applies Batch Normalization.
    discriminator.add(BatchNormalization(momentum=0.8))
    # Applies the Leaky ReLU activation function.
    discriminator.add(LeakyReLU(alpha=0.2))
    # Applies dropout.
    discriminator.add(Dropout(0.25))

    # Convolutional block 3:
    # Applies a 2D convolutional layer with 256 filters, a 3x3 kernel,
    # a stride of 1 (no downsampling), and same padding.
    discriminator.add(Conv2D(256, kernel_size=3, strides=1, padding="same"))
    # Applies Batch Normalization.
    discriminator.add(BatchNormalization(momentum=0.8))
    # Applies the Leaky ReLU activation function.
    discriminator.add(LeakyReLU(alpha=0.2))
    # Applies dropout.
    discriminator.add(Dropout(0.25))

    # Convolutional block 4:
    # Applies a 2D convolutional layer with 512 filters, a 3x3 kernel,
    # a stride of 1 (no downsampling), and same padding.
    discriminator.add(Conv2D(512, kernel_size=3, strides=1, padding="same"))
    discriminator.add(BatchNormalization(momentum=0.8))
    discriminator.add(LeakyReLU(alpha=0.2))
    # Applies dropout.
    discriminator.add(Dropout(0.25))

    # Flatten the output of the convolutional layers into a 1D vector.
    discriminator.add(Flatten())

    # Output layer:
    # Applies a dense layer with 1 unit.
    # Uses the sigmoid activation function to output a probability between 0 and 1,
    # representing whether the input image is real (closer to 1) or fake (closer to 0).
    discriminator.add(Dense(1, activation="sigmoid"))

    return discriminator

In [19]:
discriminator = build_discriminator()
discriminator.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d (ZeroPadding2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 17, 17, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 9, 9, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 9, 9, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 9, 9, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 9, 9, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 9, 9, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 9, 9, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 9, 9, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 9, 9, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 9, 9, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 9, 9, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │        41,473 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,613,889 (6.16 MB)

 Trainable params: 1,611,969 (6.15 MB)

 Non-trainable params: 1,920 (7.50 KB)